# VIneyards UAV — Thermal Sharpening — Overview & Quick Guide

✅ **Key inputs:** an Excel table (variable `excel_table`) that lists each run and provides paths for the high‑resolution reference raster and the low‑resolution thermal raster.

Purpose
- Perform spatial sharpening of UAV thermal imagery using higher‑resolution multispectral/RGB/DSM data via `pyDMS`.

Inputs (what the Excel must contain)
- Expected columns used by this notebook: `run_id`, `multiband_raster`, `tir_raster`.
- Point the `excel_table` variable to your spreadsheet; each row becomes one processing run.
- Low‑resolution thermal input can be in **Celsius or Kelvin**. The notebook auto‑detects units (median < 200 => Celsius) and converts Celsius to Kelvin (adds 273.15) and saves a `_TIR_k.tif` copy.

Important input checks (do these first)
- Confirm `CRS`, `extent`, `resolution` and `band order` of inputs.
- Verify `nodata` values and mask coverage; masks should mark *good* pixels with the `mask_values` setting (default = `1`).
- Ensure the Excel paths are valid and accessible from your environment.

Notebook structure & processing steps
1. Configuration & imports — set `excel_table`, `OUTPUT_BASE` and sharpening parameters (`useDecisionTree`, `disaggregatingTemperature`, `windowSize`, `mask_values`).
2. Read Excel & validation — load rows, validate file paths and metadata.
3. Thermal preprocessing — replace nodata → NaN, apply sentinel rules, detect units and convert Celsius → Kelvin when needed.
4. Mask creation — a `*_thermal_msk.tif` mask is created and used for quality filtering.
5. Train & sharpen — `DecisionTreeSharpener` or `NeuralNetworkSharpener` is trained and applied per row.
6. Residual analysis & correction — compute residuals, optionally correct and save outputs.
7. Export & QC — sharpened GeoTIFF (`*_TIR_sharp.tif`), residuals (`*_TIR_sharp_residual.tif`), and QC plots/metrics are written under `OUTPUT_BASE/<run_name>/`.

Outputs & QC
- Expected outputs per run: sharpened thermal TIFF, residual TIFF, mask (if created), and console/logged QC metrics.
- Check difference maps, histograms and numeric metrics (RMSE, correlation) for verification.

Quick start (3 steps)
1. Set `excel_table` to your inputs spreadsheet and confirm required columns.  
2. Update `OUTPUT_BASE` and parameters in the `Parameters` cell.  
3. Run the notebook top → bottom and inspect `outputs/<run_name>/` for results.

Troubleshooting tips
- Wrong temperature range → verify thermal units and nodata handling.  
- Misaligned rasters → reproject and ensure matching extents/resolution.  
- Artifacts/over‑sharpening → reduce window size or tuning parameters and re‑run on a small patch first.

Provenance
- Save the `Parameters` cell values and input Excel alongside results for reproducibility.

> ⚠️ Most failures come from incorrect Excel paths, mismatched CRS, or wrong thermal units — check those first.

## Example for sharpening UAV thermal imagery.

#### Note: 
This notebook showcase how to use pyDMS using UAV collected thermal and multispectral imagery. Thermal imagery has a coarser resolution than multispectral one. 

NOTE: UAV integrated multispectral and thermal sensors (like Altum and Altum PT) resample data using internal cubic convolution to match the thermal data to multispectral resolution (this has not been tested yet in this code).

#### First install these libraries in the environment (if running the first time)

We will need three images: multispectral, thermal and a mask (in this particular case, filled with the 255 value). The mask is to ensure multispectral and thermal images share a common area. Finally, we need a name for the resulting sharpenned thermal imagery.

Note that the multispectral imagery can also be a DSM, a vegetation index. The idea here is to have as much information about the surface at high resolution that a merged multispectral+DSM imagery is a good idea (not implemented here)

### The configuration of pyDMS is relativey easy:

- algorithm to use: RandomForest (useDecisionTree = True) or Neural Networks (useDecisionTree = False)
- sharpening temperature: yes (disaggregatingTemperature = True) or another data (disaggregatingTemperature = False)
- movingWindowSize is in pixels i.e. 15 means a window of 15 pixels.  an odd number is generally preferred for spatial operations. An odd window size ensures a symmetric window around the central pixel, which is important for local statistics and filtering. This symmetry allows the window to have a true center,


### Running the code...

In [5]:
import os
import time
import numpy as np

import pandas as pd

from pathlib import Path
from osgeo import gdal

import pyDMS.pyDMSUtils as utils
from pyDMS.pyDMS import DecisionTreeSharpener, NeuralNetworkSharpener
from pyDMS.pyDMS import REG_sknn_ann, REG_sklearn_ann


In [6]:
# Read input Excel table
excel_table = r"E:\AGU_2025\TSEB_files\input_generation\01_ThermalSharp_demo.xlsx"
# excel_table = r"E:\AGU_2025\TSEB_files\input_generation\01_ThermalSharpening_inputs.xlsx"
df = pd.read_excel(excel_table, header=0) # If wanting to run everything delete "skiprows" agrument
# df = pd.read_excel(excel_table, header=0, skiprows=[1,2,3,4,5,7] ) # If wanting to run everything delete "skiprows" agrument

num_runs = len(df)
display(df)
print("There are ", num_runs, "runs the model will excecute.")

,run_id,multiband_raster,tir_raster
0,dem o,E:\Github\pyDMS_AT_sac\inputs\SLM_RGBNIR_viney...,E:\AGU_2025\TSEB_files\input_generation\output...


There are  1 runs the model will excecute.


In [7]:
# User defined parameters
useDecisionTree = True # if False, a neural network sharpener will be used
disaggregatingTemperature = True # if false, it will not apply temperature radiometric transformation.
mask_values = 1 # pixels with this value in the lowResMaskFilename will be processed.
windowSize = 45 # this is the size of the moving window used for the homogeneity test. 

# Base output directory
OUTPUT_BASE = Path(r"E:\AGU_2025\TSEB_files\input_generation\outputs_tsharp_demo")
OUTPUT_BASE.mkdir(parents=True, exist_ok=True)
# Start timer
start_time = time.time()
print("Workflow started...")
# Loop to itearate over each row in the df
for i in range(len(df)):
        # print current run
        print(f"Starting run {i+1}/{num_runs} with multiband: {df['run_id'].iloc[i]}")
        highResFilename = Path(df['multiband_raster'].iloc[i]) # it can be multispectral, RGB, DSM, vegetation index.
        lowResFilename = Path(df['tir_raster'].iloc[i]) # this can be original TIR * 0 + 1
        # Validate paths
        if not highResFilename.exists():
                raise FileNotFoundError(f"Raster not found: {highResFilename}")
        if not lowResFilename.exists():
                raise FileNotFoundError(f"Raster not found: {lowResFilename}")
         # ---------------------------------------------------------------OUTPUTS------------------------------------------------------  
        # Create output folder, named after multiband file (without extension)
        # e.g., from E:\AGU_2025\TSEB_files\input_generation\DATA\RIP\RIP_720_20190504_1025_RGBNIR.tif to RIP_720_20190504_1025
        run_name = "_".join(highResFilename.stem.split("_")[:-1])
        output_dir = OUTPUT_BASE / (run_name)
        output_dir.mkdir(parents=True, exist_ok=True)
        # outputs paths
        outputFilename=os.path.join(output_dir, run_name + "_TIR_sharp_2.tif") # TIR sharpened path
        #----------------------------------------------------------------Workflow------------------------------------------------------
        # Converting lowResFilename to Kelvin. Use it only if disaggregatingTemperature is True
        
        lowResThermal = gdal.Open(lowResFilename)
        band =lowResThermal.GetRasterBand(1)
        data_LR = band.ReadAsArray().astype(float)
        # Get geotransform and projection
        gt_LR = lowResThermal.GetGeoTransform()
        proj_LR = lowResThermal.GetProjection()
        # Get NoData value
        nodata = band.GetNoDataValue()
        # Replacing NoData with NaN BEFORE converting to Kelvin
        # 1. Raster-defined NoData
        if nodata is not None:
                data_LR[data_LR == nodata] = np.nan
        # 2. Float32 sentinel NoData (very common in thermal rasters)
        FLOAT_NODATA = -3.4028235e+38
        data_LR[np.isclose(data_LR, FLOAT_NODATA, rtol=0, atol=1e+30)] = np.nan
        # 3. Fallback rule: physically impossible values
        # (using  threshold= -50
        data_LR[data_LR < -50] = np.nan

        # Converting Celsius to Kelvin 
        if np.nanmedian(data_LR) <200:
                data_LR += 273.15
                # Saving the converted Thermal image to Kelvin
                temp_lowres_path = output_dir / f"{run_name}_TIR_k.tif"
                utils.saveImg(data_LR, gt_LR, proj_LR, str(temp_lowres_path), noDataValue=None)
                lowResFilename = temp_lowres_path
        else:
                print("Thermal data already in Kelvin")
        # Creating and saving the mask 
        mask = data_LR * 0 + 1
        mask_path = output_dir / f"{run_name}_thermal_msk.tif"
        utils.saveImg(mask, gt_LR, proj_LR, str(mask_path))
        lowResMaskFilename = str(mask_path)

        commonOpts = {"highResFiles":               [highResFilename],
                  "lowResFiles":                [lowResFilename],
                  "lowResQualityFiles":         [lowResMaskFilename], # Using the created mask
                  "lowResGoodQualityFlags":     [mask_values], #this is the value of the mask that indicates good quality pixels
                  "cvHomogeneityThreshold":     0,
                  "movingWindowSize":           windowSize,
                  "disaggregatingTemperature":  [disaggregatingTemperature]}

        dtOpts =     {"perLeafLinearRegression":    True,
                        "linearRegressionExtrapolationRatio": 0.25}

        sknnOpts =   {'hidden_layer_sizes':         (10,),
                        'activation':                 'tanh'}

        nnOpts =     {"regressionType":             REG_sklearn_ann,
                        "regressorOpt":               sknnOpts}

        # start_time = time.time()

        if useDecisionTree:
                opts = commonOpts.copy()
                opts.update(dtOpts)
                disaggregator = DecisionTreeSharpener(**opts)
        else:
                opts = commonOpts.copy()
                opts.update(nnOpts)
                disaggregator = NeuralNetworkSharpener(**opts)

        print("Training regressor...")
        disaggregator.trainSharpener()
        print("Sharpening...")
        downscaledFile = disaggregator.applySharpener(highResFilename, lowResFilename)
        print("Residual analysis...")
        residualImage, correctedImage = disaggregator.residualAnalysis(downscaledFile, lowResFilename,
                                                                        lowResMaskFilename,
                                                                        doCorrection=True)
        print("Saving output...")
        highResFile = gdal.Open(highResFilename)
        if correctedImage is not None:
                outImage = correctedImage
        else:
                outImage = downscaledFile
        # outData = utils.binomialSmoother(outData)
        outFile = utils.saveImg(outImage.GetRasterBand(1).ReadAsArray(),
                                outImage.GetGeoTransform(),
                                outImage.GetProjection(),
                                outputFilename)
        residualFile = utils.saveImg(residualImage.GetRasterBand(1).ReadAsArray(),
                                        residualImage.GetGeoTransform(),
                                        residualImage.GetProjection(),
                                        os.path.splitext(outputFilename)[0] + "_residual" +
                                        os.path.splitext(outputFilename)[1])

        outFile = None
        residualFile = None
        downscaledFile = None
        highResFile = None

print(time.time() - start_time, "seconds")

Workflow started...
Starting run 1/1 with multiband: dem o
Thermal data already in Kelvin
Saved E:\AGU_2025\TSEB_files\input_generation\outputs_tsharp_demo\SLM_RGBNIR_vineyards\SLM_RGBNIR_vineyards_thermal_msk.tif
Training regressor...
Homogeneity CV threshold: 0.07
Number of training elements for is 3136 representing 100% of avaiable low-resolution data.
Homogeneity CV threshold: 0.06
Number of training elements for is 3808 representing 100% of avaiable low-resolution data.
Homogeneity CV threshold: 0.06
Number of training elements for is 3808 representing 100% of avaiable low-resolution data.
Homogeneity CV threshold: 0.09
Number of training elements for is 3808 representing 100% of avaiable low-resolution data.
Homogeneity CV threshold: 0.11
Number of training elements for is 3808 representing 100% of avaiable low-resolution data.
Homogeneity CV threshold: 0.13
Number of training elements for is 3808 representing 100% of avaiable low-resolution data.
Homogeneity CV threshold: 0.13
N